# CrossFuse v5 -- Stage B: Multimodal Training

Loads `crops-v5` (NB-A) and, if available, `sbi-visual-encoder-v5` (NB-B0),
and trains the full CrossFuse multimodal model: video / audio / sync /
fusion heads, frame-level auxiliary supervision, paired real-fake batching.

**Before launching the full run**, time ONE epoch (the timing cell below) --
it must complete in well under 60s. If it does not, the mel/MFCC cache from
NB-A is not being hit and something upstream regressed to recomputing
features per-sample.

Set `N_SEEDS=3` only for the final reported run; use `N_SEEDS=1` while
iterating on hyperparameters, to stay inside the 12h session limit.


In [ ]:
import sys, glob, os
_lib_hits = glob.glob("/kaggle/input/**/crossfuse_v5.py", recursive=True)
assert _lib_hits, ("crossfuse_v5.py not found under /kaggle/input -- attach the "
                   "crossfuse-v5-lib dataset in Add Input, then restart the session.")
sys.path.insert(0, os.path.dirname(_lib_hits[0]))
print("Using crossfuse_v5 library from:", _lib_hits[0])

import os, glob, csv, json
import numpy as np
import torch

from crossfuse_v5 import (
    CONFIG, run_training_pipeline, evaluate_deterministic, seed_everything,
)

_manifest_hits = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
assert _manifest_hits, ("manifest.csv not found under /kaggle/input -- attach the "
                        "crops-v5 dataset in Add Input, then restart the session.")
CROP_DIR = os.path.dirname(_manifest_hits[0])
print(f"Using crops from: {CROP_DIR}")

with open(os.path.join(CROP_DIR, "manifest.csv")) as f:
    MANIFEST = list(csv.DictReader(f))
for r in MANIFEST:
    r["video_label"] = int(r["video_label"]); r["audio_label"] = int(r["audio_label"])
    r["fps"] = float(r["fps"])
print(f"Manifest rows: {len(MANIFEST)}")

SBI_CKPT_CANDIDATES = glob.glob("/kaggle/input/**/sbi_visual_encoder_v5.pth", recursive=True)
CONFIG["SBI_PRETRAINED_ENCODER"] = SBI_CKPT_CANDIDATES[0] if SBI_CKPT_CANDIDATES else None
print(f"SBI pretrained encoder: {CONFIG['SBI_PRETRAINED_ENCODER'] or 'NONE -- using ImageNet init'}")


In [ ]:
# ---- Timing gate: one epoch must complete in well under 60s ----
_timing_cfg = dict(CONFIG, EPOCHS_FROZEN=1, EPOCHS_FINETUNE=0, N_SEEDS=1)
import time
t0 = time.time()
_probe_result = run_training_pipeline(MANIFEST, CROP_DIR, _timing_cfg, tag="timing_probe",
                                      seed=CONFIG["SEED"], verbose=True)
elapsed = time.time() - t0
print(f"\nOne epoch took {elapsed:.1f}s.")
if elapsed > 90:
    print("[!] Slower than expected -- check that manifest rows actually have "
          "mfcc_full/mfcc_trim/mel cached (re-run NB-A if not) before launching "
          "the full multi-seed run below.")
del _probe_result
torch.cuda.empty_cache()


In [ ]:
# ---- Main run across N_SEEDS ----
seed_results = []
for si in range(CONFIG["N_SEEDS"]):
    s = CONFIG["SEED"] + si
    res = run_training_pipeline(MANIFEST, CROP_DIR, CONFIG, tag="main", seed=s, verbose=True)
    seed_results.append(res)
    if si < CONFIG["N_SEEDS"] - 1:
        del res["model"]
        torch.cuda.empty_cache()

_main_result = seed_results[-1]
model = _main_result["model"]
checkpoint_path = _main_result["checkpoint_path"]
train_idx, val_idx, test_idx = (_main_result["train_idx"], _main_result["val_idx"],
                                _main_result["test_idx"])


def _ms(key):
    vals = [r["best_val"][key] for r in seed_results if not np.isnan(r["best_val"][key])]
    return (float(np.mean(vals)), float(np.std(vals))) if vals else (float("nan"), float("nan"))


print("\n" + "=" * 68)
print(f"  MAIN RUN across {CONFIG['N_SEEDS']} seed(s) -- VALIDATION AUC (mean +/- std)")
print("=" * 68)
for k, name in (("video", "Video head"), ("audio", "Audio head"),
                ("sync", "Sync head "), ("fusion", "Fusion head")):
    m_, s_ = _ms(k)
    print(f"  {name}: {m_:.4f} +/- {s_:.4f}")

best_metrics = {
    "n_seeds": CONFIG["N_SEEDS"],
    "per_seed": [{"seed": r["seed"], "best_epoch": r["best_epoch"],
                  **{k: r["best_val"][k] for k in ("video", "audio", "sync", "fusion",
                                                   "train_video", "train_sync")}}
                 for r in seed_results],
    "val_video_auc_mean_std": _ms("video"), "val_audio_auc_mean_std": _ms("audio"),
    "val_sync_auc_mean_std": _ms("sync"), "val_fusion_auc_mean_std": _ms("fusion"),
    "n_train": len(train_idx), "n_val": len(val_idx), "n_test": len(test_idx),
    "sbi_pretrained": CONFIG["SBI_PRETRAINED_ENCODER"] is not None,
    "config": {k: str(v) for k, v in CONFIG.items()},
}
with open("/kaggle/working/best_metrics_v5.json", "w") as f:
    json.dump(best_metrics, f, indent=2)
print(f"\nCheckpoint: {checkpoint_path}")
print("best_metrics_v5.json written. Save Version, publish /kaggle/working as "
      "'crossfuse-v5-ckpt', attach to NB-C and NB-D.")
